In [1]:
#Unified Upper_Limb_Rehabilitation Suite: Agentic AI + RAG System

In [2]:
# Goal:
# Build an AI-Powered Unified Upper Limb Rehabilitation Suite that combines
# Retrieval-Augmented Generation (RAG), Agentic AI, and intelligent tool routing
# to provide both knowledge-based rehabilitation assistance and
# action-based healthcare resource services.

# Core Capabilities:
# 1. Answer technical questions about shoulder, elbow, and wrist rehabilitation stimulators
#    using PDF manuals and rehabilitation documents
# 2. Compare upper-limb rehabilitation systems using retrieved document context
# 3. Explain rehabilitation workflows, calibration procedures, therapy guidance,
#    and safety precautions
# 4. Check rehabilitation resource availability
# 5. Book rehabilitation resources for patients
# 6. Provide rehabilitation facility information and therapy unit descriptions
# 7. Maintain conversational memory for agent-driven rehabilitation interactions

# Architecture:
# - RAG for technical knowledge retrieval from upper-limb rehabilitation manuals
# - LangGraph for agent orchestration and conversational memory management
# - Intelligent Router for stable production execution
# - SQLite for rehabilitation resource management
# - Gradio for interactive healthcare user interface

# Modules:
# 1. Knowledge Layer (RAG)
# 2. Action Layer (SQL Tools)
# 3. Agent Layer (LangGraph + Memory)
# 4. Routing Layer (Intent Router)
# 5. User Interface Layer (Gradio)

# Final Outcome:
# An AI-Powered Unified Upper Limb Rehabilitation Assistant capable of answering
# technical rehabilitation questions, comparing rehabilitation stimulators,
# managing rehabilitation resources, supporting patient bookings,
# and enabling interactive rehabilitation support for shoulder,
# elbow, and wrist therapy systems.

In [3]:
# Install Dependencies
%%capture

!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-core
!pip install -q langchain-groq
!pip install -q langgraph
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q gradio

In [4]:
# Import Libraries

import os
import sqlite3
import gradio as gr
import getpass

# PDF Loading and Chunking
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector Database
from langchain_community.vectorstores import FAISS

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Prompting and RAG Chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Tools
from langchain_core.tools import tool

# Messages
from langchain_core.messages import HumanMessage

# LLM
from langchain_groq import ChatGroq

# LangGraph (Agentic Layer + Memory)
from langgraph.graph import StateGraph
from langgraph.prebuilt import ToolNode, create_react_agent, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

# Typing
from typing import TypedDict, Annotated

/tmp/ipykernel_514/1855543027.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
# Load Upper Limb Rehabilitation PDFs

loaders = [
    PyPDFLoader("3D Robotic Elbow Rehabilitation Stimulator.pdf"),
    PyPDFLoader("3D Robotic Shoulder Rehabilitation Stimulator V2.pdf"),
    PyPDFLoader("3D Robotic Wrist Rehabilitation Stimulator.pdf")
]

docs = []

for loader in loaders:
    docs.extend(loader.load())

print(f"Loaded {len(docs)} pages successfully")

Loaded 12 pages successfully


In [7]:
# Split & Chunk Documents

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n", "\n", "Step", "Calibration", "."]
)

chunks = text_splitter.split_documents(docs)

print(f"Created {len(chunks)} chunks")

Created 16 chunks


In [8]:
# Created Embeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Created Vector Database

vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)

print("Vector database created successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database created successfully


In [9]:
import getpass
import os
# Set Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Initialize LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_retries=2
)

print("LLM initialized successfully")

Enter your Groq API Key: ··········
LLM initialized successfully


In [10]:
# Retriever (MMR)

retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20}
)

print("Retriever ready")

Retriever ready


In [11]:
# Advanced RAG Prompt

template = """
You are an advanced rehabilitation engineering assistant specialized in:

- robotic rehabilitation systems
- upper-limb rehabilitation
- shoulder rehabilitation
- elbow rehabilitation
- wrist rehabilitation
- telekinematic rehabilitation workflows
- rehabilitation biomechanics
- therapy guidance
- rehabilitation safety procedures
- rehabilitation monitoring
- web-based rehabilitation systems
- rehabilitation engineering architectures
- interactive rehabilitation feedback systems

Use ONLY the provided rehabilitation manual context.

Instructions:
- Answer technically and professionally.
- Focus on rehabilitation workflows, rehabilitation engineering,
  therapy guidance, and clinical utility.
- Explain rehabilitation procedures clearly and systematically.
- If remote rehabilitation or tele-rehabilitation is mentioned,
  explain it as a telekinematic rehabilitation workflow.
- When architecture, workflow, monitoring, or system-design questions are asked,
  provide structured diagrammatic-style explanations using:
  system blocks,
  workflow pipelines,
  monitoring layers,
  rehabilitation stages,
  feedback loops,
  control-flow structures,
  and clinical interaction flow.
- Generate clean architecture-style textual representations when appropriate.
- Explain rehabilitation monitoring, progress tracking,
  safety workflows, therapy guidance,
  and rehabilitation control mechanisms clearly.
- Avoid generic healthcare explanations.
- Do not invent hardware or sensors unless explicitly mentioned in the context.
- If the answer is partially available,
  answer only from the available rehabilitation context.

Context:
{context}

Question:
{question}

Technical Answer:
"""

In [12]:
# RAG Chain

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | ChatPromptTemplate.from_template(template)
    | llm
    | StrOutputParser()
)

print("RAG Chain ready")

RAG Chain ready


In [13]:
DB_FILE = "upper_limb_rehabilitation_suite.db"

def init_db():
    conn = sqlite3.connect(DB_FILE, check_same_thread=False)
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS resources")
    cur.execute("DROP TABLE IF EXISTS bookings")

    cur.execute("""
    CREATE TABLE resources (
        resource_id INTEGER PRIMARY KEY,
        resource_name TEXT,
        status TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE bookings (
        booking_id INTEGER PRIMARY KEY,
        patient_name TEXT,
        resource_id INTEGER,
        FOREIGN KEY(resource_id) REFERENCES resources(resource_id)
    )
    """)

    cur.executemany("""
    INSERT INTO resources (resource_id, resource_name, status)
    VALUES (?, ?, ?)
    """, [
        (1, "Robotic Shoulder Rehabilitation Stimulator", "available"),
        (2, "Robotic Elbow Rehabilitation Stimulator", "available"),
        (3, "Robotic Wrist Rehabilitation Stimulator", "available"),
        (4, "Upper Limb Physiotherapy Unit", "available"),
        (5, "Motion Analysis and Rehabilitation Lab", "available")
    ])

    conn.commit()
    conn.close()

init_db()

print("Upper Limb Rehabilitation DB initialized")

Upper Limb Rehabilitation DB initialized


In [14]:
def sql_query(query):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute(query)
    result = cursor.fetchall()
    conn.commit()
    conn.close()
    return result

In [15]:
sql_query("SELECT * FROM resources")

[(1, 'Robotic Shoulder Rehabilitation Stimulator', 'available'),
 (2, 'Robotic Elbow Rehabilitation Stimulator', 'available'),
 (3, 'Robotic Wrist Rehabilitation Stimulator', 'available'),
 (4, 'Upper Limb Physiotherapy Unit', 'available'),
 (5, 'Motion Analysis and Rehabilitation Lab', 'available')]

In [16]:
# Tool 1: Knowledge Tool (RAG)

@tool
def rehab_knowledge_tool(question: str) -> str:
    """
    Use ONLY for technical questions about upper-limb rehabilitation stimulators,
    calibration, sensors, therapy workflow, safety precautions,
    rehabilitation exercises, and simulator manuals.

    Covers:
    - Shoulder rehabilitation systems
    - Elbow rehabilitation systems
    - Wrist rehabilitation systems

    Do NOT use for booking or resource availability questions.
    """
    print("CALLING: rehab_knowledge_tool()")

    return rag_chain.invoke(question)

In [17]:
# Availability Tool

@tool
def check_availability_tool() -> str:
    """
    Use ONLY for checking available upper-limb rehabilitation resources.

    Do NOT use for technical simulator/manual questions.
    Do NOT use for booking resources.
    """
    print("CALLING: check_availability_tool()")

    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    cursor.execute(
        "SELECT resource_id, resource_name FROM resources WHERE status='available'"
    )

    rows = cursor.fetchall()

    conn.close()

    if not rows:
        return "No upper-limb rehabilitation resources available."

    return "\n".join(
        [f"Resource {r[0]}: {r[1]}" for r in rows]
    )

In [20]:
# Booking Tool

@tool
def book_resource_tool(patient_name: str, resource_id: int) -> str:
    """
    Use ONLY when the user explicitly requests to book
    an upper-limb rehabilitation resource.

    Required:
    - patient_name
    - resource_id

    Do NOT use for technical/manual questions.
    Do NOT use for availability checking.
    """
    print("CALLING: book_resource_tool()")

    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    cursor.execute(
        "SELECT * FROM resources WHERE resource_id=? AND status='available'",
        (resource_id,)
    )

    resource = cursor.fetchone()

    if not resource:
        conn.close()
        return f"Resource {resource_id} is not available."

    cursor.execute(
        "INSERT INTO bookings (patient_name, resource_id) VALUES (?, ?)",
        (patient_name, resource_id)
    )

    cursor.execute(
        "UPDATE resources SET status='occupied' WHERE resource_id=?",
        (resource_id,)
    )

    conn.commit()
    conn.close()

    return (
        f"Upper-limb rehabilitation resource {resource_id} "
        f"booked successfully for {patient_name}."
    )

In [21]:
conn = sqlite3.connect(DB_FILE)

cursor = conn.cursor()

cursor.execute("SELECT * FROM resources")

rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

(1, 'Robotic Shoulder Rehabilitation Stimulator', 'available')
(2, 'Robotic Elbow Rehabilitation Stimulator', 'available')
(3, 'Robotic Wrist Rehabilitation Stimulator', 'available')
(4, 'Upper Limb Physiotherapy Unit', 'available')
(5, 'Motion Analysis and Rehabilitation Lab', 'available')


In [30]:
# Register Tools

tools = [
    rehab_knowledge_tool,
    check_availability_tool,
    book_resource_tool
]

system_message = """
You are an AI-Powered Unified Upper Limb Rehabilitation Suite Assistant.

Available tools:
1. rehab_knowledge_tool → technical/manual rehabilitation questions
2. check_availability_tool → upper-limb rehabilitation resource availability
3. book_resource_tool → rehabilitation resource booking

Supported Rehabilitation Systems:
- Shoulder Rehabilitation Stimulator
- Elbow Rehabilitation Stimulator
- Wrist Rehabilitation Stimulator

Rules:

1. For technical/manual rehabilitation questions:
   - Use rehab_knowledge_tool
   - Answer using retrieved rehabilitation document context
   - Return the tool answer directly to the user

2. For rehabilitation resource availability:
   - Use check_availability_tool
   - Return the tool answer directly to the user

3. For rehabilitation resource booking:
   - Use book_resource_tool only if patient_name and resource_id are present
   - If details are missing, ask the user for the missing information

4. If the user asks multiple rehabilitation-related questions in one message:
   - Use the required tools
   - Combine outputs clearly and professionally

5. Always provide the actual tool results as the final answer.

6. Focus on rehabilitation workflows, calibration procedures,
   therapy guidance, safety precautions, simulator operation,
   and upper-limb rehabilitation support.
"""

In [26]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [31]:
def healthcare_suite_router(user_query):
    query = user_query.lower()

    # Booking intent
    if "book" in query:
        return "Please provide patient name and resource ID."

    # Availability intent
    elif "available" in query or "availability" in query:
        return check_availability_tool.invoke({})

    # Resource / facility information intent
    elif (
        "motion analysis" in query
        or "physiotherapy unit" in query
        or "resource" in query
    ):
        return """
Motion Analysis and Rehabilitation Lab:
A specialized upper-limb rehabilitation facility used to evaluate
arm movement, joint coordination, therapy progression,
range of motion, and biomechanical rehabilitation performance.

The lab supports shoulder, elbow, and wrist rehabilitation assessment,
therapy planning, movement tracking, and rehabilitation monitoring.
"""

    # Technical/manual intent
    else:
        return rehab_knowledge_tool.invoke(
            {"question": user_query}
        )

In [32]:
# Rebuilding Unified Upper Limb Rehabilitation Agent
# Agentic Layer (LangGraph Version)

agent = create_react_agent(
    llm,
    tools,
    prompt=system_message
)

graph_builder = StateGraph(State)

# Add agent node
graph_builder.add_node("agent", agent)

# Add tool node
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

# Add routing logic
graph_builder.add_conditional_edges(
    "agent",
    tools_condition
)

# Return to agent after tool execution
graph_builder.add_edge("tools", "agent")

# Entry point
graph_builder.set_entry_point("agent")

# Conversational Memory
memory = MemorySaver()

# Compile graph
graph = graph_builder.compile(
    checkpointer=memory
)

print("Unified Upper Limb Rehabilitation Agent rebuilt successfully")

Unified Upper Limb Rehabilitation Agent rebuilt successfully


/tmp/ipykernel_514/2015510632.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [33]:
def healthcare_suite_router(user_query):
    query = user_query.lower()

    # Booking intent
    if "book" in query:
        return "Please provide patient name and resource ID."

    # Availability intent
    elif "available" in query or "availability" in query:
        return check_availability_tool.invoke({})

    # Resource / rehabilitation facility information intent
    elif (
        "motion analysis" in query
        or "physiotherapy" in query
        or "rehabilitation lab" in query
        or "resource" in query
    ):
        return """
Motion Analysis and Rehabilitation Lab:

A specialized upper-limb rehabilitation facility designed to evaluate
arm movement, joint coordination, rehabilitation performance,
range of motion, and biomechanical therapy progression.

The lab supports:
- Shoulder rehabilitation assessment
- Elbow rehabilitation therapy
- Wrist rehabilitation monitoring
- Motion tracking and rehabilitation analysis
- Therapy planning and rehabilitation workflow evaluation
"""

    # Technical/manual intent
    else:
        return rehab_knowledge_tool.invoke(
            {"question": user_query}
        )

In [34]:
# LangGraph Testing

result = healthcare_suite_router(
    "How does the robotic shoulder rehabilitation stimulator work?"
)

print(result)

CALLING: rehab_knowledge_tool()
The 3D Robotic Shoulder Rehabilitation Simulator is a web-based, interactive musculoskeletal rehabilitation system designed for patients recovering from shoulder surgery or trauma. It utilizes advanced computer vision and a dynamic 3D kinematic model to provide real-time feedback and guidance for safe and effective rehabilitation.

**System Architecture:**

The system consists of the following components:

1. **User Interface (UI):** A mobile-friendly interface with a light blue gradient background, accessible through a standard web browser on a mobile phone or laptop.
2. **3D Kinematic Model:** A dynamic 3D model of the shoulder complex (glenohumeral joint) that simulates flexion and abduction protocols.
3. **Robotic Arm:** An elbow flexion/extension robotic arm that assists patients in performing movements within a controlled digital environment.
4. **Computer Vision:** Advanced computer vision technology that analyzes joint mobility and healing indica

In [35]:
#LangGraph Testing(Comparing)
result = healthcare_suite_router(
    "Compare the elbow and wrist rehabilitation stimulators."
)

print(result)

CALLING: rehab_knowledge_tool()
**Rehabilitation Workflow Comparison: Elbow vs. Wrist Rehabilitation Stimulators**

The 3D Robotic Elbow Rehabilitation Stimulator and the 3D Robotic Wrist Rehabilitation Stimulator are two distinct interactive musculoskeletal rehabilitation systems designed for orthopedic rehabilitation. Both systems utilize advanced computer vision and a 3D simulation environment to provide real-time feedback and guidance for patients recovering from surgery or trauma.

**System Architecture Comparison**

Both systems consist of the following components:

1. **Patient Interface**: A web-based application accessible through a standard web browser on a mobile phone or laptop.
2. **Robotic Arm**: Assists patients in executing controlled movements to restore joint mobility.
3. **Computer Vision**: Analyzes joint mobility and healing indicators to provide real-time feedback.
4. **Feedback System**: Provides color-coded feedback and progress tracking to prevent overstrain an

In [36]:
# LangGraph Multi-Tool Testing

query = """
Can you list the available upper-limb rehabilitation resources
and explain the safety precautions for the shoulder rehabilitation stimulator
from the manual?
"""

result = healthcare_suite_router(query)

print(result)

CALLING: check_availability_tool()
Resource 1: Robotic Shoulder Rehabilitation Stimulator
Resource 2: Robotic Elbow Rehabilitation Stimulator
Resource 3: Robotic Wrist Rehabilitation Stimulator
Resource 4: Upper Limb Physiotherapy Unit
Resource 5: Motion Analysis and Rehabilitation Lab


In [37]:
import gradio as gr

# =========================
# RESPONSE FUNCTION
# =========================

def respond(message, history):

    try:

        # Gradio message
        if isinstance(message, dict):
            user_message = message.get("content", "")
        else:
            user_message = str(message)

        print(f"USER QUERY: {user_message}")

        # Route Query
        result = healthcare_suite_router(user_message)

        print(f"RESULT: {result}")

        return {
            "role": "assistant",
            "content": str(result)
        }

    except Exception as e:

        print(f"ERROR: {e}")

        return {
            "role": "assistant",
            "content": f"System Error: {str(e)}"
        }


# =========================
# SIMPLE HEADER
# =========================

description_html = """
<div style="text-align:center;">
    <h3>Built by Dr. Lakshmi Gandi</h3>
</div>
"""


# =========================
# GRADIO CHAT INTERFACE
# =========================

demo = gr.ChatInterface(

    fn=respond,

    type="messages",

    title="🏥 Unified Upper Limb Rehabilitation Suite",

    description=description_html,

    examples=[

        "How does the robotic shoulder rehabilitation stimulator work?",

        "Compare the elbow and wrist rehabilitation stimulators.",

        "What safety precautions are mentioned in the manuals?",

        "Which upper-limb rehabilitation resources are available?",

        "Book resource 2 for Ravi",

        "What is the Motion Analysis and Rehabilitation Lab?"
    ]
)


# =========================
# LAUNCH APPLICATION
# =========================

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6d061d075103afcec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
